# Stage 0 — Hypothesis v5: BTC Overnight Sessions + N-day MAX

**Date locked:** 2026-05-24  
**Author:** Anthony Chung  
**Version:** v5.0  
**Predecessors:** v1, v2 (Stage 2 fails), v3, v4 (Stage 4 fails — both TSMOM-family)  
**Status:** ACTIVE

---

## Why v5 (not v4 tweak)

Per v4's pre-commitment: no v5 may attempt another TSMOM tweak. v5 must use a fundamentally different mechanism.

v5 satisfies this:
- **Single asset** (BTC only, not multi-asset like v3/v4)
- **Time-of-day filter** (specific session boundaries, not weekly rebalance)
- **Breakout signal** (at N-day MAX, not trailing return)
- **Short hold** (overnight/weekend hold, not 1-week rebalance)

Reference: [QuantPedia (2024) — How To Profitably Trade Bitcoin's Overnight Sessions](https://quantpedia.com/how-to-profitably-trade-bitcoins-overnight-sessions/) by Cyril Dujava.

## Thesis

Bitcoin's intra-day returns are unevenly distributed: most positive risk-adjusted return historically occurs during **overnight (US-equity-market closed) hours**, particularly when BTC enters those hours at a new local high. Trading only these specific hours with a momentum-confirming filter (at N-day MAX) should produce a high-Sharpe-per-unit-of-exposure strategy.

## Market Inefficiency Targeted

**Asymmetric session returns + breakout momentum**:

1. Crypto markets are 24/7, but flow is heavily influenced by US equity-market participants who are active 14:30-21:00 UTC (NYSE 9:30-16:00 EST).
2. When US equity participants leave the market (after NYSE close), price discovery shifts to Asian/EU participants with different positioning.
3. If BTC closes US session AT a 10-day high, it suggests strong demand state; this positioning persists overnight.
4. The overnight session captures the residual buy pressure before US re-opens.
5. By exiting at US morning open, we avoid the noisier intra-day phase.

QuantPedia's analysis (2024) using 9 years of hourly BTC data found this pattern statistically significant with excellent Sharpe and low MaxDD.

## Theoretical / Empirical Support

| Reference | Contribution |
|-----------|--------------|
| [QuantPedia (2024) — Overnight Bitcoin](https://quantpedia.com/how-to-profitably-trade-bitcoins-overnight-sessions/) | Direct 9-year backtest of this exact strategy |
| [QuantPedia (2024) — Revisiting Trend-following](https://quantpedia.com/revisiting-trend-following-and-mean-reversion-strategies-in-bitcoin/) | Confirms N-day MAX strategy as effective; 10-day optimal |
| Yang (2018) | Momentum WORKS on crypto |
| Hong & Stein (1999) — "A Unified Theory of Underreaction, Momentum Trading, and Overreaction" | Theoretical foundation: information diffuses through investor groups → trending continues |

Session-based anomalies are well-documented across asset classes (Lou et al 2019 for equities; Branch & Ma 2012 for FX). The crypto application is novel-ish but consistent with the framework.

## Strategy Specification

### Universe
- **BTCUSDT only** (single-asset, simpler test of the time-of-day mechanism)
- Multi-coin extension belongs to Stage 6

### Trading sessions (UTC, derived from NYSE timing)
- Three entry windows per week (per QuantPedia spec):
  - **Friday 21:00 UTC** → Monday 14:00 UTC (weekend hold, ~65 hrs)
  - **Monday 21:00 UTC** → Tuesday 14:00 UTC (overnight ~17 hrs)
  - **Tuesday 21:00 UTC** → Wednesday 14:00 UTC (overnight ~17 hrs)
- 21:00 UTC = 4pm EST ≈ NYSE close
- 14:00 UTC = 10am EDT / 9am EST ≈ NYSE morning hours

### Entry signal
At each session start (21:00 UTC on Fri/Mon/Tue):
```
max_recent = max(close[t - lookback_days * 24 : t + 1])
if close[t] >= max_recent * (1 - breakout_proximity_pct):
    enter LONG at close[t]   (= open[t+1] approx, crypto 24/7)
else:
    no trade this session
```

### Exit
Always at the corresponding morning open (14:00 UTC). No stop loss, no take profit — pure session strategy.

### Position sizing
Equal $ per trade (1× equity); not levered.

## Locked Parameter Search Space

| Parameter | Search values | Notes |
|-----------|---------------|-------|
| `lookback_days` | [5, 10, 15, 20] | QuantPedia tested 10/20/30/40/50; we test similar plus shorter |
| `breakout_proximity_pct` | [0.000, 0.005, 0.010] | strict MAX vs slight relaxation |
| `entry_hour_utc` | [20, 21, 22] | Around NYSE close (4pm EST) |
| `exit_hour_utc` | [13, 14, 15] | Around NYSE morning |

**Total trials:** 4 × 3 × 3 × 3 = **108**

Days locked at Fri+Mon+Tue per QuantPedia spec.

## Falsifiable Predictions

| # | Test | Threshold | Stage |
|---|------|-----------|-------|
| 1 | At least 1 of 108 IS trials has PF > 1.0 | required | 2 |
| 2 | OOS-1 PF retention | ≥ 70% | 3 |
| 3 | Walk-forward profitable months | ≥ 80% | 4 |
| 4 | Holdout PF | > 1.3 | 5 |
| 5 | Holdout Sharpe | > 1.0 | 5 |
| 6 | Holdout MaxDD | < 30% | 5 |
| 7 | Deflated Sharpe (N=108) | > 0 | 5 |
| 8 | Multi-coin replication | ≥ 4 of 6 pass tests 4-6 | 6 |

**Specific prediction**: this is the FIRST short-hold strategy we test (12-65 hrs vs 1 week for v3/v4). Walk-forward should benefit from higher trade frequency (more samples per month → less variance per measurement).

## Pre-Commitment Statement

> I have not examined any v5 backtest prior to this commit. The parameter ranges are based on QuantPedia's 2024 published spec with reasonable variations.
>
> If v5 fails, no further parameter tweaking. v6 (if attempted) must again use a fundamentally different mechanism.
>
> This is the 5th hypothesis. Each prior failure is documented in `results/POSTMORTEM.md`. I am explicitly not allowing the prior failures to compromise rigor — each new hypothesis gets the same 6-stage treatment.
>
> — Anthony Chung, 2026-05-24